# LLM agents & orchestration

LLM agents wrap model calls in planning, memory, tools, and termination logic.

An agent is a small state machine around model calls: plan from state and memory, act with a tool, observe the result, and decide whether to continue or stop. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

random.seed(9)
np.random.seed(9)

RUNG_NAMES = ["D1", "D2", "D3", "D4", "D5"]


def softmax(logits):
    values = np.array(logits, dtype=float)
    shifted = values - values.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def tokenize(text):
    clean = "".join(ch.lower() if ch.isalnum() else " " for ch in text)
    return [tok for tok in clean.split() if tok]


def cosine(a, b):
    left = np.array(a, dtype=float)
    right = np.array(b, dtype=float)
    denom = np.linalg.norm(left) * np.linalg.norm(right)
    if denom == 0:
        return 0.0
    return float(np.dot(left, right) / denom)


def bow_vector(text, vocab):
    counts = Counter(tokenize(text))
    return np.array([counts.get(term, 0) for term in vocab], dtype=float)


def preview_ladder(ladder):
    for rung in ladder:
        name = rung["name"]
        size = len(rung.get("tasks", rung.get("items", rung.get("queries", []))))
        note = rung.get("note", "")
        print(f"{name}: size={size} {note}")
        sample_key = "tasks" if "tasks" in rung else "items" if "items" in rung else "queries"
        print(rung[sample_key][0])


## The concept, built once

The lesson formula is $s_{t+1}=\mathrm{observe}(\mathrm{act}(\mathrm{plan}(s_t,m_t)))$. We represent planning as dependency resolution and action choice as a scored controller. Memory retrieval is deliberately limited, because every retrieved item becomes state that later actions may trust.

In [ ]:

def ready_steps(steps, completed):
    ready = []
    for step, deps in steps.items():
        if step in completed:
            continue
        if all(dep in completed for dep in deps):
            ready.append(step)
    return ready


def retrieve_memory(query, memories, k):
    vocab = sorted(set(tokenize(query) + tokenize(" ".join(memories))))
    query_vec = bow_vector(query, vocab)
    scored = []
    for memory in memories:
        score = cosine(query_vec, bow_vector(memory, vocab))
        scored.append((score, memory))
    scored.sort(reverse=True)
    return [memory for score, memory in scored[:k]]


The reusable loop executes ready steps, updates state through observations, stops after success, and can aggregate agent votes. The worked numbers assert the lesson's exact planning, memory, termination, and voting quantities.

In [ ]:

def agent_loop(task, max_steps=6, use_stop_rule=True):
    completed = set(task.get("completed", []))
    trace = []
    for step_index in range(max_steps):
        ready = ready_steps(task["steps"], completed)
        if not ready:
            break
        scored_ready = [(task["controller_scores"].get(step, 0.0), step) for step in ready]
        score, action = max(scored_ready)
        memories = retrieve_memory(action + " " + task["goal"], task["memories"], task["memory_k"])
        completed.add(action)
        trace.append({"action": action, "score": score, "memories": memories})
        if use_stop_rule and action == task["success_step"]:
            break
    success = task["success_step"] in completed
    continuation = 1 - task["termination_probability"] if success else 1.0
    votes = task["votes"] + [task["judge_vote"]]
    majority = sum(votes) / len(votes)
    return {"trace": trace, "success": success, "continuation": continuation, "majority": majority}


lesson_agent_task = {
    "goal": "finish workflow D",
    "steps": {"A": [], "B": [], "C": ["A", "B"], "D": ["C"]},
    "controller_scores": {"A": 0.2, "B": 0.6, "C": 0.4, "D": 0.9},
    "memories": [f"memory {i}" for i in range(10)],
    "memory_k": 3,
    "termination_probability": 0.8,
    "votes": [1, 1, 0],
    "judge_vote": 1,
    "success_step": "D",
}

lesson_ready = ready_steps(lesson_agent_task["steps"], set())
lesson_result = agent_loop(lesson_agent_task)

assert len(lesson_ready) == 2
assert max([0.2, 0.6, 0.4]) == 0.6
assert lesson_agent_task["memory_k"] / len(lesson_agent_task["memories"]) == 0.3
assert round(1 - lesson_agent_task["termination_probability"], 3) == 0.2
assert round(lesson_result["majority"], 3) == 0.750
print(lesson_result)


## The dataset ladder

Build D1-D5 inline so the notebook is self-contained in Colab. Each rung adds scale or a realistic failure mode while keeping CPU-only toy data.

In [ ]:

def make_agent_ladder():
    base = {"goal": "publish checked answer", "steps": {"A": [], "B": [], "C": ["A", "B"], "D": ["C"]}, "controller_scores": {"A": 0.2, "B": 0.6, "C": 0.4, "D": 0.9}, "memories": ["A gathers facts", "B runs calculator", "C drafts answer", "D verifies final", "distractor old state", "tool budget note", "format rule", "prior success", "irrelevant", "archive"], "memory_k": 3, "termination_probability": 0.8, "votes": [1, 1, 0], "judge_vote": 1, "success_step": "D"}
    d1 = [base]
    d2 = d1 + [{**base, "goal": "file support ticket", "controller_scores": {"A": 0.7, "B": 0.5, "C": 0.6, "D": 0.8}}]
    d3 = d2 + [{**base, "goal": "answer with distracting memories", "memories": base["memories"] + ["wrong completed D", "stale unsafe action"]}]
    d4 = d3 + [{**base, "goal": "lookup price convert units then answer", "steps": {"lookup": [], "convert": [], "cite": ["lookup", "convert"], "stop": ["cite"]}, "controller_scores": {"lookup": 0.7, "convert": 0.8, "cite": 0.6, "stop": 0.9}, "success_step": "stop"}]
    d5 = d4 + [{**base, "goal": "long multi-tool workflow with tempting post-success planning", "steps": {"plan": [], "act": ["plan"], "verify": ["act"], "final": ["verify"], "extra": ["final"]}, "controller_scores": {"plan": 0.5, "act": 0.7, "verify": 0.8, "final": 0.9, "extra": 1.0}, "success_step": "final"}]
    return [{"name": "D1", "tasks": d1, "note": "one prompt"}, {"name": "D2", "tasks": d2, "note": "few-shot tasks"}, {"name": "D3", "tasks": d3, "note": "distractor memories"}, {"name": "D4", "tasks": d4, "note": "real-style tool workflow"}, {"name": "D5", "tasks": d5, "note": "nontermination risk"}]


agent_ladder = make_agent_ladder()
preview_ladder(agent_ladder)


## Run the same method across D1-D5

The metric follows the plan: task success, retrieval recall, unsupported rate, benchmark accuracy, or safety pass-rate depending on the topic.

In [ ]:

agent_rows = []
for rung in agent_ladder:
    completions = []
    step_counts = []
    for task in rung["tasks"]:
        result = agent_loop(task, max_steps=8, use_stop_rule=True)
        completions.append(float(result["success"]))
        step_counts.append(len(result["trace"]))
    agent_rows.append({"rung": rung["name"], "metric": sum(completions) / len(completions), "steps": sum(step_counts) / len(step_counts)})

for row in agent_rows:
    print(row)


## Results visualization

The closing figure uses small multiples for the per-rung artifact and one summary curve across D1-D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for index, rung in enumerate(agent_ladder):
    result = agent_loop(rung["tasks"][-1], max_steps=8, use_stop_rule=True)
    actions = [item["action"] for item in result["trace"]]
    axes[0, index].bar(range(len(actions)), [1] * len(actions), color="seagreen")
    axes[0, index].set_xticks(range(len(actions)))
    axes[0, index].set_xticklabels(actions, rotation=45)
    axes[0, index].set_title(rung["name"])

axes[1, 0].plot([row["rung"] for row in agent_rows], [row["metric"] for row in agent_rows], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_ylabel("completion rate")
axes[1, 0].set_title("completion vs rung")
for blank in axes[1, 1:]:
    blank.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Reproduce the named D5 failure, then turn on the fix and compare the behavior.

In [ ]:

d5_agent = agent_ladder[-1]["tasks"][-1]
without_stop = agent_loop(d5_agent, max_steps=8, use_stop_rule=False)
with_stop = agent_loop(d5_agent, max_steps=8, use_stop_rule=True)
print("without explicit stop", [item["action"] for item in without_stop["trace"]])
print("with explicit stop", [item["action"] for item in with_stop["trace"]])


## Evaluate it + Practice

- Compare the planned metric with a no-skill baseline such as answering without tools, retrieval, claims, intervals, or guardrails.
- Sanity-check one hand-computable lesson number before trusting the ladder.
- Ablate the key idea and verify the metric drops or the failure signal rises.
- Watch failure signals: retry loops, irrelevant evidence, hidden unsupported claims, noisy rank changes, or unsafe tool actions.

Practice prompts:
- Add a dependency branch E that depends on B and compare ready-step counts.
- Lower memory_k from 3 to 1 and inspect which action loses useful context.
- Create a voting tie and add a judge rule that breaks it explicitly.
